In [ ]:
import numpy as np
import scanpy as sc
import multigrate as mtg
import muon

In [ ]:
adata = sc.read_h5ad("./Data/RNA_ATAC/BMMC/BMMC.h5ad")

adata_gex = adata[:, adata.var['modality'] == "Gene Expression"]
adata_atac = adata[:, adata.var['modality'] == "Peaks"]
print(adata_gex)
print(adata_atac)

AnnData object with n_obs × n_vars = 69249 × 129921
    obs: 'cell_type', 'batch', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'DonorID', 'DonorAge', 'DonorGender'
    var: 'gene_id', 'modality'
    uns: 'ATAC_gene_activity_var_names', 'dataset_id', 'genome', 'organism'
    obsm: 'ATAC_gene_activity', 'ATAC_lsi_full', 'ATAC_lsi_red', 'ATAC_umap', 'GEX_X_pca', 'GEX_X_umap'
    layers: 'counts'

In [4]:

sc.pp.normalize_total(adata_gex, target_sum=1e4)
sc.pp.log1p(adata_gex)
sc.pp.highly_variable_genes(adata_gex, n_top_genes=4000, batch_key='batch')
adata_gex = adata_gex[:, adata_gex.var.highly_variable].copy()

sc.pp.log1p(adata_atac)


In [ ]:
adata = mtg.data.organize_multiome_anndatas(
    adatas = [[adata_gex], [adata_atac]],            # a list of anndata objects per modality, RNA-seq always goes first
    layers = [['counts'], [None]],     # if need to use data from .layers, if None use .X
)
mtg.model.MultiVAE.setup_anndata(
    adata,
    categorical_covariate_keys=["batch"],
    rna_indices_end = 4000,
)
model = mtg.model.MultiVAE(
    adata,
    losses=['nb', 'mse'],
   # z_dim=60,
)
model.train()

model.plot_losses()
model.get_latent_representation()

In [ ]:
np.save('Multigrate_BMMC.npy', adata.obsm['latent'])